In [1]:

# ======================================================================
# HELIX CLOUD | GENOMIC INTELLIGENCE & CLOUD RELIABILITY PLATFORM
# ONE-CELL COMPLETE APPLICATION
# ======================================================================

import sys, subprocess, importlib.util, tempfile, os, warnings
warnings.filterwarnings("ignore")

# 1. Install dependencies
deps = {
    "gradio": "gradio",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "plotly": "plotly",
    "networkx": "networkx"
}
missing = [pip for module, pip in deps.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])

import gradio as gr
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
from datetime import datetime

# ======================================================================
# 2. INFRASTRUCTURE DEFINITION
# ======================================================================

SERVICES = {
    "Variant API":       (35, "API", 0.95),
    "QC Worker":         (42, "Compute", 0.85),
    "Feature Engine":    (54, "Compute", 0.90),
    "ML Inference":      (62, "AI", 1.00),
    "Variant Database":  (48, "Database", 0.95),
    "Object Storage":    (30, "Storage", 0.70),
    "Job Queue":         (25, "Messaging", 0.75),
    "Report Generator":  (38, "Compute", 0.65),
    "Monitoring":        (22, "Observability", 0.80)
}
EDGES = [
    ("Variant API", "QC Worker"),
    ("QC Worker", "Feature Engine"),
    ("Feature Engine", "ML Inference"),
    ("ML Inference", "Variant Database"),
    ("ML Inference", "Report Generator"),
    ("Variant API", "Job Queue"),
    ("Job Queue", "QC Worker"),
    ("Variant Database", "Object Storage"),
    ("Report Generator", "Object Storage"),
    ("Monitoring", "ML Inference")
]
GRAPH = nx.DiGraph()
for name, (cpu, tier, criticality) in SERVICES.items():
    GRAPH.add_node(name, tier=tier, criticality=criticality)
GRAPH.add_edges_from(EDGES)

FEATURES = ["PrimateAI-3D", "SpliceAI", "PhyloP"]
CLOUD_FEATURES = ["CPU (%)", "Latency (ms)", "Error Rate (%)"]
STATUS_COLORS = {
    "HEALTHY": "#22c55e",
    "DEGRADED": "#f59e0b",
    "FAILED": "#ef4444",
    "RECOVERED": "#38bdf8"
}

# ======================================================================
# 3. SYNTHETIC GENOMIC DATA
# ======================================================================

def make_demo_data(n=1200, seed=42):
    rng = np.random.default_rng(seed)
    primate = rng.beta(2.0, 2.5, n)
    splice = rng.beta(1.5, 4.0, n)
    phylo = rng.normal(2.2, 2.0, n).clip(-5, 10)

    # Artificial labels created for demonstration only
    latent = (
        2.6 * primate + 3.4 * splice + 0.11 * phylo
        + rng.normal(0, 0.8, n)
    )
    probability = 1 / (1 + np.exp(-(latent - np.median(latent))))
    target = rng.binomial(1, probability)

    return pd.DataFrame({
        "Variant_ID": [f"DEMO_{i:05d}" for i in range(n)],
        "Gene": rng.choice(
            ["GENE_A", "GENE_B", "GENE_C", "GENE_D", "GENE_E"], n
        ),
        "PrimateAI-3D": primate,
        "SpliceAI": splice,
        "PhyloP": phylo,
        "target": target
    })

# ======================================================================
# 4. DATA LOADING AND VALIDATION
# ======================================================================

def load_genomic_data(path):
    if path:
        df = pd.read_csv(path)
        source = "Uploaded CSV"
    else:
        df = make_demo_data()
        source = "Synthetic demonstration dataset"

    # Normalize feature column names
    normalized = {
        str(c).strip().lower().replace("_", "").replace("-", ""): c
        for c in df.columns
    }
    aliases = {
        "primateai3d": "PrimateAI-3D",
        "spliceai": "SpliceAI",
        "phylop": "PhyloP"
    }

    rename = {}
    for normalized_name, canonical in aliases.items():
        if canonical not in df.columns and normalized_name in normalized:
            rename[normalized[normalized_name]] = canonical
    df = df.rename(columns=rename)

    if "target" not in df.columns:
        for candidate in ["label", "class", "pathogenic"]:
            if candidate in df.columns:
                df = df.rename(columns={candidate: "target"})
                break

    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing feature columns: {missing}. "
            f"Expected {FEATURES}."
        )
    if "target" not in df.columns:
        raise ValueError(
            "A supervised dataset must have a binary 'target' column "
            "(0 or 1)."
        )

    for feature in FEATURES + ["target"]:
        df[feature] = pd.to_numeric(df[feature], errors="coerce")

    df = df.dropna(subset=["target"]).copy()
    df = df[df["target"].isin([0, 1])].copy()
    df["target"] = df["target"].astype(int)

    if len(df) < 30 or df["target"].nunique() < 2:
        raise ValueError(
            "Need at least 30 labeled rows and both target classes."
        )
    if df[FEATURES].isna().all(axis=None):
        raise ValueError("All genomic feature values are missing.")

    return df, source

# ======================================================================
# 5. TRAIN AND EVALUATE MODEL
# ======================================================================

def fit_model(df):
    X, y = df[FEATURES], df["target"]
    if y.value_counts().min() < 2:
        raise ValueError("Each class needs at least two examples.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=180,
            max_depth=8,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    }

    importance = pd.DataFrame({
        "Feature": FEATURES,
        "Importance": model.named_steps["classifier"].feature_importances_
    }).sort_values("Importance", ascending=False)

    evaluation = pd.DataFrame({
        "Actual": y_test.to_numpy(),
        "Predicted": pred,
        "Positive-class score": probs.round(4)
    })

    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    return model, metrics, importance, evaluation, cm

# ======================================================================
# 6. GENOMIC ANALYSIS DASHBOARD
# ======================================================================

def genomic_analysis(path):
    try:
        df, source = load_genomic_data(path)
        model, metrics, importance, evaluation, cm = fit_model(df)

        summary = f"""
        <div class="cards">
          <div class="card"><small>DATA ROWS</small>
          <h2>{len(df):,}</h2><p>{source}</p></div>
          <div class="card"><small>TARGET 1</small>
          <h2>{int((df.target == 1).sum()):,}</h2></div>
          <div class="card"><small>TARGET 0</small>
          <h2>{int((df.target == 0).sum()):,}</h2></div>
          <div class="card"><small>MISSING FEATURE VALUES</small>
          <h2>{int(df[FEATURES].isna().sum().sum())}</h2></div>
        </div>
        """
        metric_html = "<div class='cards'>" + "".join(
            f"<div class='card'><small>{k.upper()}</small>"
            f"<h2>{v:.3f}</h2></div>"
            for k, v in metrics.items()
        ) + "</div>"

        importance_fig = px.bar(
            importance.sort_values("Importance"),
            x="Importance", y="Feature", orientation="h",
            title="Random Forest Feature Importance",
            template="plotly_dark"
        )
        importance_fig.update_layout(
            paper_bgcolor="#0b1220", plot_bgcolor="#0b1220", height=350
        )

        cm_fig = go.Figure(go.Heatmap(
            z=cm,
            x=["Predicted 0", "Predicted 1"],
            y=["Actual 0", "Actual 1"],
            text=cm, texttemplate="%{text}",
            colorscale="Blues"
        ))
        cm_fig.update_layout(
            title="Held-Out Test Confusion Matrix",
            template="plotly_dark",
            paper_bgcolor="#0b1220", plot_bgcolor="#0b1220",
            height=350
        )

        return (
            summary, metric_html, importance_fig, cm_fig,
            df.head(30), evaluation, importance,
            "Training complete. Results describe model performance on a held-out split."
        )

    except Exception as exc:
        empty = go.Figure()
        msg = f"Analysis error: {exc}"
        return (
            f"<p>{msg}</p>", "", empty, empty,
            pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), msg
        )

# ======================================================================
# 7. VARIANT SCORE DEMO
# ======================================================================

def score_variant(primate, splice, phylo):
    df = make_demo_data(1000, 42)
    model, metrics, importance, evaluation, cm = fit_model(df)
    sample = pd.DataFrame([{
        "PrimateAI-3D": float(primate),
        "SpliceAI": float(splice),
        "PhyloP": float(phylo)
    }])
    score = float(model.predict_proba(sample)[0, 1])
    label = int(model.predict(sample)[0])

    return pd.DataFrame([{
        **sample.iloc[0].to_dict(),
        "Model positive-class score": round(score, 4),
        "Model prediction": label,
        "Disclaimer": "Synthetic training data; not a clinical classification."
    }])

# ======================================================================
# 8. CLOUD DIGITAL TWIN + ISOLATION FOREST
# ======================================================================

def simulate_cloud(failure, traffic, recovery, seed):
    rng = np.random.default_rng(int(seed))
    rows = []
    affected = set()
    if failure != "None":
        affected = nx.descendants(GRAPH, failure)
        affected.add(failure)

    for service, (base_cpu, tier, criticality) in SERVICES.items():
        cpu = float(np.clip(base_cpu * float(traffic) + rng.normal(0, 4), 5, 100))
        latency = float(max(5, 40 + cpu * 1.7 + rng.normal(0, 10)))
        errors = float(max(0, rng.normal(0.4, 0.2)))
        status = "HEALTHY"

        if service == failure:
            cpu = min(100, cpu + 20)
            latency += 200
            errors += 6
            if recovery:
                cpu *= 0.65
                latency *= 0.60
                errors *= 0.25
                status = "RECOVERED"
            else:
                status = "FAILED"

        elif service in affected:
            cpu = min(100, cpu + 10)
            latency += 65
            errors += 2
            if recovery:
                cpu *= 0.85
                latency *= 0.80
                errors *= 0.50
                status = "RECOVERED"
            else:
                status = "DEGRADED"

        rows.append({
            "Service": service, "Tier": tier,
            "CPU (%)": round(cpu, 2),
            "Latency (ms)": round(latency, 2),
            "Error Rate (%)": round(errors, 2),
            "Criticality": criticality, "Status": status
        })

    df = pd.DataFrame(rows)

    # Isolation Forest trained on simulated normal telemetry
    baseline_rng = np.random.default_rng(int(seed) + 100)
    baseline = pd.DataFrame({
        "CPU (%)": baseline_rng.normal(40, 12, 1000).clip(5, 85),
        "Latency (ms)": baseline_rng.normal(110, 25, 1000).clip(10, 250),
        "Error Rate (%)": baseline_rng.normal(0.5, 0.3, 1000).clip(0, 2)
    })
    detector = IsolationForest(
        n_estimators=150, contamination=0.10, random_state=int(seed)
    )
    detector.fit(baseline)
    df["ML Anomaly"] = detector.predict(df[CLOUD_FEATURES]) == -1
    df["Anomaly Score"] = -detector.decision_function(df[CLOUD_FEATURES])

    centrality = nx.betweenness_centrality(GRAPH)
    df["Centrality"] = df["Service"].map(centrality).fillna(0)
    df["Risk Score"] = (
        df["CPU (%)"] / 100 * 0.30
        + df["Error Rate (%)"].clip(upper=10) / 10 * 0.35
        + df["Latency (ms)"].clip(upper=1000) / 1000 * 0.20
        + df["Criticality"] * 0.10
        + df["Centrality"] * 0.05
    ).round(3)

    # Topology
    pos = nx.spring_layout(GRAPH, seed=42, k=1.1)
    ex, ey = [], []
    for u, v in GRAPH.edges():
        ex += [pos[u][0], pos[v][0], None]
        ey += [pos[u][1], pos[v][1], None]
    edge_trace = go.Scatter(
        x=ex, y=ey, mode="lines",
        line=dict(color="#334155", width=1.5),
        hoverinfo="none", showlegend=False
    )

    node_x, node_y, labels, colors, hover = [], [], [], [], []
    for _, row in df.iterrows():
        name = row["Service"]
        node_x.append(pos[name][0])
        node_y.append(pos[name][1])
        labels.append(name)
        colors.append(
            "#ef4444" if row["ML Anomaly"]
            else "#38bdf8" if row["Status"] == "RECOVERED"
            else "#22c55e"
        )
        hover.append(
            f"{name}<br>CPU: {row['CPU (%)']}%"
            f"<br>Latency: {row['Latency (ms)']} ms"
            f"<br>Anomaly: {row['ML Anomaly']}"
            f"<br>Status: {row['Status']}"
        )

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode="markers+text",
        text=labels, textposition="bottom center",
        textfont=dict(size=9, color="#dbeafe"),
        hovertext=hover, hoverinfo="text",
        marker=dict(size=24, color=colors,
                    line=dict(width=1, color="#e2e8f0")),
        showlegend=False
    )
    topology = go.Figure([edge_trace, node_trace])
    topology.update_layout(
        title="Genomic Processing Infrastructure",
        template="plotly_dark", height=500,
        paper_bgcolor="#0b1220", plot_bgcolor="#0b1220",
        xaxis=dict(visible=False), yaxis=dict(visible=False)
    )

    cpu_fig = px.bar(
        df.sort_values("CPU (%)", ascending=False),
        x="Service", y="CPU (%)", color="Status",
        title="Simulated Service CPU Utilization",
        template="plotly_dark"
    )
    cpu_fig.update_layout(
        height=400, paper_bgcolor="#0b1220",
        plot_bgcolor="#0b1220", xaxis_tickangle=-35
    )

    anomalies = int(df["ML Anomaly"].sum())
    unhealthy = int(df["Status"].isin(["FAILED", "DEGRADED"]).sum())
    mean_risk = round(float(df["Risk Score"].mean()), 3)

    cards = f"""
    <div class="cards">
      <div class="card"><small>ML ANOMALIES</small><h2>{anomalies}</h2></div>
      <div class="card"><small>UNHEALTHY SERVICES</small><h2>{unhealthy}</h2></div>
      <div class="card"><small>MEAN RISK SCORE</small><h2>{mean_risk}</h2></div>
      <div class="card"><small>SIMULATED RECOVERY</small>
      <h2>{'ON' if recovery else 'OFF'}</h2></div>
    </div>
    """

    ranked = df.sort_values("Risk Score", ascending=False)
    notes = (
        f"Highest heuristic risk: {ranked.iloc[0]['Service']}. "
        "Risk ranking is not proof of causality. Cloud data is synthetic."
    )
    report = pd.DataFrame([{
        "Timestamp": datetime.now().isoformat(timespec="seconds"),
        "Injected Failure": failure,
        "Traffic Multiplier": traffic,
        "Recovery Enabled": recovery,
        "ML Anomalies": anomalies,
        "Unhealthy Services": unhealthy,
        "Mean Risk Score": mean_risk,
        "Data Source": "Synthetic simulation"
    }])

    return cards, topology, cpu_fig, df, ranked, notes, report

# ======================================================================
# 9. CSV DOWNLOADS
# ======================================================================

def write_csv(df, filename):
    path = os.path.join(tempfile.gettempdir(), filename)
    df.to_csv(path, index=False)
    return path

# ======================================================================
# 10. INTERACTIVE DASHBOARD
# ======================================================================

CSS = """
.gradio-container {
    background:#070b14!important;
    color:#e5edf8!important;
    max-width:1550px!important;
}
#hero {
    padding:25px;
    border:1px solid #26364e;
    border-radius:20px;
    background:linear-gradient(120deg,#101b30,#11101e,#14243a);
    margin-bottom:14px;
}
#hero h1 {font-size:36px;color:#f8fafc}
#hero p {color:#bdcbe0}
.cards {
    display:grid;
    grid-template-columns:repeat(auto-fit,minmax(145px,1fr));
    gap:10px;
}
.card {
    background:#101827;
    border:1px solid #293950;
    border-radius:14px;
    padding:14px;
}
.card small {font-size:10px;color:#94a3b8;letter-spacing:1px}
.card h2 {font-size:25px;color:#7dd3fc;margin:8px 0}
.card p {font-size:11px;color:#9caec4}
"""

with gr.Blocks(
    title="HELIX CLOUD | Genomic Intelligence",
    theme=gr.themes.Soft(),
    css=CSS
) as app:

    gr.HTML("""
    <div id="hero">
      <h1>🧬 HELIX CLOUD</h1>
      <p>Genomic Intelligence · Machine Learning · Cloud Digital Twin ·
      Infrastructure Anomaly Detection</p>
      <p><b>MISSION:</b> Explore genomic data analysis and the reliability of
      the computing infrastructure that processes it.</p>
    </div>
    """)

    with gr.Tabs():
        with gr.Tab("🧬 Genomic Intelligence"):
            gr.Markdown(
                "Upload a labeled CSV or use the synthetic demo data. "
                "Required numeric features: PrimateAI-3D, SpliceAI, PhyloP. "
                "The target column must be binary (0 or 1)."
            )
            genomic_file = gr.File(
                label="Optional labeled CSV",
                file_types=[".csv"],
                type="filepath"
            )
            genomic_button = gr.Button("▶ Train and analyze", variant="primary")
            genomic_summary = gr.HTML()
            genomic_metrics = gr.HTML()
            with gr.Row():
                importance_plot = gr.Plot()
                confusion_plot = gr.Plot()
            genomic_preview = gr.Dataframe(label="Dataset preview", interactive=False)
            evaluation_table = gr.Dataframe(label="Held-out predictions", interactive=False)
            importance_table = gr.Dataframe(label="Feature importance", interactive=False)
            genomic_notes = gr.Textbox(label="Analysis notes", interactive=False)

            genomic_button.click(
                genomic_analysis,
                inputs=[genomic_file],
                outputs=[
                    genomic_summary, genomic_metrics,
                    importance_plot, confusion_plot,
                    genomic_preview, evaluation_table,
                    importance_table, genomic_notes
                ]
            )

        with gr.Tab("🔬 Variant Score Demo"):
            gr.Markdown(
                "This score uses a model trained on artificial data. "
                "It is not a clinical classification."
            )
            with gr.Row():
                p_input = gr.Slider(0, 1, value=0.7, step=0.01, label="PrimateAI-3D")
                s_input = gr.Slider(0, 1, value=0.6, step=0.01, label="SpliceAI")
                phy_input = gr.Slider(-5, 10, value=2.5, step=0.1, label="PhyloP")
            score_button = gr.Button("Score example variant", variant="primary")
            score_output = gr.Dataframe(interactive=False)
            score_button.click(
                score_variant,
                inputs=[p_input, s_input, phy_input],
                outputs=[score_output]
            )

        with gr.Tab("☁️ Cloud Digital Twin"):
            gr.Markdown(
                "Inject a simulated service failure and compare recovery on/off. "
                "No real cloud infrastructure is accessed."
            )
            with gr.Row():
                with gr.Column(scale=1):
                    failure_input = gr.Dropdown(
                        ["None"] + list(SERVICES.keys()),
                        value="ML Inference", label="Injected failure"
                    )
                    traffic_input = gr.Slider(
                        0.5, 3, value=1.5, step=0.25,
                        label="Traffic multiplier"
                    )
                    recovery_input = gr.Checkbox(
                        value=True, label="Simulate recovery"
                    )
                    seed_input = gr.Number(
                        value=42, precision=0, label="Scenario seed"
                    )
                    cloud_button = gr.Button(
                        "▶ Run cloud simulation", variant="primary"
                    )
                with gr.Column(scale=3):
                    cloud_cards = gr.HTML()

            with gr.Tabs():
                with gr.Tab("Dependency Map"):
                    cloud_topology = gr.Plot()
                with gr.Tab("CPU Monitoring"):
                    cloud_cpu = gr.Plot()
                with gr.Tab("Service Inventory"):
                    cloud_inventory = gr.Dataframe(interactive=False)
                    cloud_inventory_download = gr.DownloadButton("⬇ Download inventory CSV")
                with gr.Tab("Risk Ranking"):
                    cloud_ranking = gr.Dataframe(interactive=False)
                    cloud_notes = gr.Textbox(interactive=False)
                with gr.Tab("Incident Report"):
                    cloud_report = gr.Dataframe(interactive=False)
                    cloud_report_download = gr.DownloadButton("⬇ Download report CSV")

            cloud_outputs = [
                cloud_cards, cloud_topology, cloud_cpu,
                cloud_inventory, cloud_ranking, cloud_notes, cloud_report
            ]
            cloud_inputs = [
                failure_input, traffic_input, recovery_input, seed_input
            ]

            cloud_button.click(
                simulate_cloud, inputs=cloud_inputs, outputs=cloud_outputs
            )
            cloud_inventory_download.click(
                lambda f, t, r, s: write_csv(
                    simulate_cloud(f, t, r, s)[3],
                    "helix_cloud_inventory.csv"
                ),
                inputs=cloud_inputs, outputs=cloud_inventory_download
            )
            cloud_report_download.click(
                lambda f, t, r, s: write_csv(
                    simulate_cloud(f, t, r, s)[6],
                    "helix_cloud_report.csv"
                ),
                inputs=cloud_inputs, outputs=cloud_report_download
            )

    gr.Markdown("""
    ---
    **Limitations and responsible use**
    - Built-in genomic data and labels are synthetic, not patient data.
    - Uploaded datasets must contain labeled examples for supervised training.
    - Feature importance is not SHAP and does not prove causality.
    - Variant scores are not clinical diagnoses or medical advice.
    - Cloud metrics, anomalies, and recovery are simulated.
    """)

print("HELIX CLOUD is launching...")
app.launch(share=True, debug=False)


HELIX CLOUD is launching...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://631e2ea93636278970.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
